# machine-learning_018

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (91).ipynb`

| Field | Value |
|---|---|
| Section | `machine_learning` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 14 |
| Detected functions | — |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install kaggle catboost

In [ ]:
from google.colab import files
files.upload()  # Use this to upload kaggle.json

In [ ]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle competitions download -c jane-street-real-time-market-data-forecasting

In [ ]:
!unzip jane-street-real-time-market-data-forecasting.zip -d ./jane_street_data

In [ ]:
import pandas as pd
import numpy as np
import datetime
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

In [ ]:
import pandas as pd
data = pd.read_parquet('/content/extracted_data/train.parquet/partition_id=0/part-0.parquet')
data

In [ ]:
data = pd.DataFrame()
for i in range(2):
    partition = pd.read_parquet(f'/content/jane_street_data/train.parquet/partition_id={i}/part-0.parquet')
    data = pd.concat([data, partition])
    del partition
data

In [ ]:
!pip install --upgrade xgboost

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error

# Assuming the dataset is loaded into a DataFrame named 'data'
# Replace 'data' with your actual DataFrame variable name
# Split into X (features), y (target), and sample weights
X = data.filter(regex='^feature_').values
y = data['responder_6']  # Replace 'target' with your actual column name for y
weights = data['weight']  # Replace 'weight' with your actual column name for weights

# Split data into training and testing sets (include weights)
X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, weights, test_size=0.1, random_state=42
)


In [ ]:
from sklearn.metrics import r2_score
model_xgb = XGBRegressor(objective='reg:squarederror', eval_metric='rmse', random_state=42)
model_xgb.fit(
    X_train, y_train, sample_weight=w_train
)
y_pred_xgb = model_xgb.predict(X_train)
r2_xgb = r2_score(y_train, y_pred_xgb)
print(f"R2 score (XGBoost): {r2_xgb:.4f}")